# EDA Notebook

## 1. Importar librerías necesarias
Importar pandas, numpy, matplotlib y seaborn para el análisis y visualización de datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

sns.set_theme(context='notebook', style='whitegrid')

## 2. Carga del archivo CSV
Ruta del archivo .csv y carga en DataFrame de pandas.

In [ ]:
file_path = '../data/raw/2024_Accidentalidad.csv'
df = pd.read_csv(file_path, sep=';', encoding='utf-8')

## 3. Visualización de las primeras filas del dataset
Muestra de las primeras filas del DataFrame usando head() para obtener una vista general de los datos.

In [ ]:
df.head()

## 4. Exploración rápida de la información general del dataset
Utilizando los métodos info() y shape para mostrar información sobre el número de filas, columnas y tipos de datos.

In [ ]:
df.info()
print(f"\nNúmero de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

## 5. Describir variables numéricas y categóricas
Usar describe() para variables numéricas y value_counts() para variables categóricas.

In [ ]:
# Variables numéricas
df.describe()

In [ ]:
# Variables categóricas
columnas = [
    "distrito",
    "tipo_accidente",
    "estado_meteorológico",
    "tipo_vehiculo",
    "tipo_persona",
    "rango_edad",
    "sexo",
    "positiva_alcohol"
]
resultados = {col: df[col].value_counts() for col in columnas}

for col, valores in resultados.items():
    print(f"\nResultados para la categoría:")
    print(valores)

## 6. Visualización de valores nulos y duplicados
Identificar y visualizar la cantidad de valores nulos y filas duplicadas en el dataset.

In [ ]:
# Valores nulos
print(f"Valores Nulos:\n{df.isnull().sum()}")

# Filas/Observaciones duplicadas
print(f"\nFilas Duplicadas: {df.duplicated().sum()}")

## 7. Limpieza de Dataset
- Eliminar variables que no son necesarias para el análisis.
- Eliminar posibles espacios o caracteres especiales en el nombre de las variables. Estandarizar.
- Eliminar observaciones/filas duplicadas.
- Cambiar variables por tipo de datos acorde a su contenido.
- Agregar variable 'fecha_hora' de tipo datetime y eliminar variables separadas 'fecha' y 'hora'.
- Unificar criterio de observaciones/filas localizadas con diferentes sinónimos para "Desconocido" en el dataset por 'unknown'.
- Imputación en tres variables categóricas que representan porcentajes bajos por 'unknown' y la moda.
- Modificar tipo de dato de str/object a categoria para el ahorro de memoria.



In [ ]:
# Generar copia del dataset sucio para la limpieza
df_clean = df.copy()

# Eliminar variables que no son necesarias para el análisis.
df_clean = df_clean.drop(columns=['num_expediente', 'localizacion', 'numero', 'cod_distrito', 'cod_lesividad', 'lesividad', 'coordenada_x_utm', 'coordenada_y_utm'], errors="ignore")

# Eliminar posibles espacios o caracteres especiales en el nombre de las variables. Estandarizar.
df_clean.columns = [c.strip().lower().replace(' ', '_') for c in df_clean.columns]

# Eliminar observaciones/filas duplicadas.
df_clean = df_clean.drop_duplicates()
print(f"Filas Duplicadas: {df_clean.duplicated().sum()}")

# Cambiar variables por tipo de datos acorde a su contenido.
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'], errors='coerce', dayfirst=True)
df_clean['hora'] = pd.to_datetime(df_clean['hora'], format="%H:%M:%S", errors='coerce')
df_clean['positiva_alcohol'] = (df_clean['positiva_alcohol'].map({'S':True, 'N': False}).fillna(False).astype("bool"))
df_clean['positiva_droga'] = (df_clean['positiva_droga'].fillna(0)).astype('bool')


#Agregar variable 'fecha_hora' de tipo datetime y eliminar variables separadas 'fecha' y 'hora'.
df_clean.insert(0, 'fecha_hora',
                df_clean['fecha'] + (df_clean['hora'] - df_clean['hora'].dt.normalize()))
df_clean = df_clean.drop(columns=['fecha', 'hora'], errors="ignore")

# Unificar criterio de observaciones/filas localizadas con diferentes sinónimos para "Desconocido" en el dataset por "unknown".
df_clean['estado_meteorológico'] = df_clean['estado_meteorológico'].astype('string').str.strip().str.lower().replace({'se desconoce': 'unknown'})
df_clean['rango_edad'] = df_clean['rango_edad'].astype('string').str.strip().str.lower().replace({'desconocido': 'unknown', 'más de 74 años': 'mayor a 74 años', 'menor de 5 años': 'menor a 5 años'})
df_clean['sexo'] = df_clean['sexo'].astype('string').str.strip().str.lower().replace({'desconocido': 'unknown'})

# Imputación en tres variables categóricas que representan porcentajes bajos de valores nulos por 'unknown', se evita imputación con la moda por tratarse de categorias importantes y de impacto.
#print(f'\nValores Nulos:\n{df_clean.isnull().sum()}')
#print((df_clean[['tipo_accidente', 'estado_meteorológico', 'tipo_vehiculo']].isna().mean()) * 100) 
df_clean[['tipo_accidente', 'estado_meteorológico', 'tipo_vehiculo']] = df_clean[['tipo_accidente', 'estado_meteorológico', 'tipo_vehiculo']].fillna('unknown')
print(f'\nValores Nulos:\n{df_clean.isnull().sum()}')

# Modificar tipo de dato de str/object a categoria para el ahorro de memoria.
mem_antes = df.memory_usage(deep=True).sum()
columnas_categoria = [
    "distrito",
    "tipo_accidente",
    "estado_meteorológico",
    "tipo_vehiculo",
    "tipo_persona",
    "rango_edad",
    "sexo"
]
df_clean[columnas_categoria] = df_clean[columnas_categoria].astype("category")

## Memoria después
mem_despues = df_clean.memory_usage(deep=True).sum()
print('\nMemoria antes:', mem_antes)
print('Memoria después :', mem_despues)
print('Ahorro de Memoria(%):', round((mem_antes - mem_despues) / mem_antes * 100, 2))

# Muestra del dataset después de la limpieza
print('\nDataset después de la limpieza:', df_clean.shape)
display(df_clean.head())

## 8. Features y Tranformaciones 
- Se crea una copia del dataset limpio para generar nuevas variables sin alterar los datos originales.
- Se extraen variables temporales desde fecha_hora: mes, día de la semana y hora.
- Se construye la variable temporada para analizar patrones de riesgo según la época del año.
- Se calcula el porcentaje de accidentes con alcohol positivo por distrito. Este porcentaje se asigna a cada registro para enriquecer el análisis.
- Se agrupan los rangos de edad en categorías simplificadas: niños, jóvenes, adultos y mayores.
- Se crea la variable grupo_etario para facilitar análisis comparativos por edad.
- Se convierte grupo_etario a tipo categórico para optimizar memoria y análisis.

Estas transformaciones permiten identificar patrones temporales, geográficos y demográficos de riesgo.

In [ ]:
# Generar copia del dataset limpio para agregar variables temporales y transformaciones
df_features = df_clean.copy()

# Variables Temporales
df_features['mes'] = df_features['fecha_hora'].dt.month
df_features['dia_semana'] = df_features['fecha_hora'].dt.day_name()
df_features['hora'] = df_features['fecha_hora'].dt.hour

## Variable que permita analizar patrones de riesgo por época del año.
def asignar_temporada(mes):
    if mes in [12, 1, 2]:
        return 'invierno'
    elif mes in [3, 4, 5]:
        return 'primavera'
    elif mes in [6, 7, 8]:
        return 'verano'
    elif mes in [9, 10, 11]:
        return 'otoño'
    else:
        return 'unknown'

df_features['temporada'] = df_features['mes'].apply(asignar_temporada)

## Calcular porcentaje de accidentes con alcohol positivo por distrito
pct_alcohol = (
    df_features.groupby('distrito')['positiva_alcohol']
      .mean()                           # obtiene la media → proporción de True
      .mul(100)                         # convierte a porcentaje
      .round(2)                         # redondea a 2 decimales
)

# Inicializar en 0
df_features['pct_alcohol_por_distrito'] = 0.0

# Asignar porcentaje sólo a los True
df_features.loc[df_features['positiva_alcohol'] == True,
                'pct_alcohol_por_distrito'] = (
    df_features['distrito'].map(pct_alcohol)
)

## Variable que permita contrastar si ciertos grupos etarios presentan más riesgo con drogas o alcohol. 
# Definir el mapeo de rangos originales a categorías simplificadas
mapa_grupo_etario = {
    'menor a 5 años': 'niños',
    'de 6 a 9 años': 'niños',
    'de 10 a 14 años': 'niños',
    'de 15 a 17 años': 'jóvenes',
    'de 18 a 20 años': 'jóvenes',
    'de 21 a 24 años': 'jóvenes',
    'de 25 a 29 años': 'jóvenes',
    'de 30 a 34 años': 'adultos',
    'de 35 a 39 años': 'adultos',
    'de 40 a 44 años': 'adultos',
    'de 45 a 49 años': 'adultos',
    'de 50 a 54 años': 'adultos',
    'de 55 a 59 años': 'adultos',
    'de 60 a 64 años': 'adultos',
    'de 65 a 69 años': 'mayores',
    'de 70 a 74 años': 'mayores',
    'mayor a 74 años': 'mayores',
    'unknown': 'unknown'
}
# Aplicar el mapeo
df_features['grupo_etario'] = df_features['rango_edad'].map(mapa_grupo_etario)

# Convertir a tipo categoría
df_features['grupo_etario'] = df_features['grupo_etario'].astype('category')

display(df_features[['fecha_hora','mes','dia_semana','hora','temporada', 'distrito','pct_alcohol_por_distrito','rango_edad', 'grupo_etario']].head(10))

## 8.1 Asserts
Verificaciones  

In [178]:
assert df_features['distrito'].isna().sum() == 0
assert df_features['grupo_etario'].isna().sum() == 0
assert ((df_features['pct_alcohol_por_distrito'] >= 0) & 
        (df_features['pct_alcohol_por_distrito'] < 6)).all()
print('Checks básicos superados')

Checks básicos superados


## 9. Análisis univariado de variables numéricas
Realizar histogramas y boxplots para analizar la distribución de las variables numéricas.

In [ ]:
df_viz = df_features.copy()

# Distribución de accidentes por hora del día — Histograma numérico
plt.figure(figsize=(8,4))
sns.histplot(df_viz['hora'], bins=24, color='darkorange')
plt.title("Distribución de accidentes por hora del día")
plt.xlabel("Hora")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

# Boxplot del porcentaje de alcohol positivo por distrito — Boxplot numérico/categórico
plt.figure(figsize=(12,6))
sns.boxplot(data=df_viz, x='distrito', y='pct_alcohol_por_distrito', color='lightcoral')
plt.title("% de accidentes con alcohol positivo por distrito")
plt.xlabel("Distrito")
plt.ylabel("Porcentaje de alcohol positivo")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 10. Análisis univariado de variables categóricas
Realizar gráficos de barras para analizar la frecuencia de las variables categóricas.

In [ ]:
# Distribución de accidentes por distrito top 10 — Barras verticales
plt.figure(figsize=(10,5))
df_viz['distrito'].value_counts().head(10).plot(kind='bar', color='steelblue')
plt.title("Top 10 distritos con más accidentes")
plt.xlabel("Distrito")
plt.ylabel("Número de accidentes")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Proporción de accidentes por temporada — Gráfico de pastel
plt.figure(figsize=(6,6))
df_viz['temporada'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', startangle=90, cmap='Pastel2'
)
plt.ylabel('')
plt.title("Proporción de accidentes por temporada")
plt.tight_layout()
plt.show()

## 11. Análisis bivariado entre variables
Explorar relaciones entre variables numéricas y categóricas mediante scatterplots y tablas cruzadas.

In [ ]:
#Comparativa alcohol y droga por grupo etario — Barras lado a lado
comp = (
    df_viz.groupby('grupo_etario')[['positiva_alcohol','positiva_droga']]
      .mean().mul(100).round(2)
      .reset_index()
)
plt.figure(figsize=(8,5))
x = range(len(comp))
plt.bar(x, comp['positiva_alcohol'], width=0.4, label='Alcohol +', color='tomato')
plt.bar([i + 0.4 for i in x], comp['positiva_droga'], width=0.4, label='Droga +', color='teal')
plt.xticks([i + 0.2 for i in x], comp['grupo_etario'])
plt.title("Porcentaje de positivos (alcohol/droga) por grupo etario")
plt.xlabel("Grupo etario")
plt.ylabel("Porcentaje (%)")
plt.legend()
plt.tight_layout()
plt.show()

# Accidentes por grupo etario y tipo de accidente — Gráfico de barras agrupadas
plt.figure(figsize=(12,6))
sns.countplot(data=df_viz, x='grupo_etario', hue='tipo_accidente', 
              order=['niños','jóvenes','adultos','mayores'], palette='tab10')
plt.title("Distribución de tipo de accidente por grupo etario")
plt.xlabel("Grupo etario")
plt.ylabel("Número de accidentes")
plt.legend(title='Tipo de accidente', bbox_to_anchor=(1.05,1))
plt.tight_layout()
plt.show()


## 12. Visualización de correlaciones
Calcular y visualizar la matriz de correlación entre variables numéricas usando un heatmap.

In [ ]:
# Matriz de correlaciones numéricas — Heatmap global de correlación
plt.figure(figsize=(6,6))
sns.heatmap(df_viz.select_dtypes('number').corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matriz de correlaciones numéricas")
plt.tight_layout()
plt.show()

# Mapa de calor (heatmap) de hora vs día de la semana — Cruce temporal de dos variables categóricas
pivot_hora_dia = df_viz.pivot_table(index='dia_semana', columns='hora', 
                                         values='distrito', aggfunc='count')
plt.figure(figsize=(12,6))
sns.heatmap(pivot_hora_dia, cmap='YlOrRd')
plt.title("Frecuencia de accidentes por hora y día de la semana")
plt.xlabel("Hora del día")
plt.ylabel("Día de la semana")
plt.tight_layout()
plt.show()